In [ ]:
###--- Load libraries and set the location path for analysis data ---###

In [ ]:
# Environment setup
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.io
import matplotlib as mpl
import batchglm.api as glm
import diffxpy.api as de
import decoupler as dc

from matplotlib import rcParams
import bbknn
import os
import sys
import scipy
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scrublet as scr
import scipy.stats as stats

In [ ]:
sc.settings.verbosity = 2  # show logging output
sc.settings.dir = "scRNA_Glyco/"
sc.settings.autosave = True  # save figures, do not show them
sc.settings.figdir = "scRNA_Glyco/figure"
sc.settings.set_figure_params(dpi=800, format="pdf", dpi_save=800)  # set sufficiently high resolution for saving 400dpi

In [ ]:
###--- Load pre-filtering data ---###

In [ ]:
# Data Loading
adata = sc.read("sc_allcells_annotation_global.h5ad")
adata_celltypist_ly = sc.read("sc_lymphoid_clustering.h5ad")
adata_celltypist_my = sc.read("sc_myeloid_clustering.h5ad")

In [ ]:
# Create sub_cell_type_spec annotationd
adata_celltypist_ly.obs['comb'] = 'SubLymphoid_' + adata_celltypist_ly.obs['leiden'].astype(str)
adata_celltypist_my.obs['comb'] = 'SubMyeloid_' + adata_celltypist_my.obs['leiden'].astype(str)

# Find the indices of observations that satisfy the condition
selected_indices_my = adata.obs['bc_wells'].isin(adata_celltypist_my.obs['bc_wells'])
selected_indices_ly = adata.obs['bc_wells'].isin(adata_celltypist_ly.obs['bc_wells'])

# Assign values based on selected indices
adata.obs.loc[selected_indices_my, 'cell_type_spec'] = adata_celltypist_my.obs['comb']
adata.obs.loc[selected_indices_ly, 'cell_type_spec'] = adata_celltypist_ly.obs['comb']

# Assign values based on original annotation
adata.obs['cell_type_spec'] = adata.obs['cell_type_spec'].fillna(adata.obs['cell_type'])

del adata_celltypist_ly
del adata_celltypist_my
np.unique(adata.obs['cell_type_spec'], return_counts=True)

In [ ]:
###--- Basic Preparation  ---###
del adata.uns["log1p"]
# Start with Raw data
adata.X = adata.layers["counts"].copy()

# List of samples
comparison_id= ['CAR1','CAR2','CAR3','Tr2DG1','Tr2DG2','Tr2DG3','TrTUN1','TrTUN2','TrTUN3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = adata.obs['sample'].isin(comparison_id)
adata = adata[boolean_mask, :]

In [ ]:
# Normalization
sc.pp.normalize_total(adata, target_sum=1e4)
adata.layers["norm10k"] = adata.X

# Logarithmize the data:
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X

In [ ]:
###--- Ligand-receptor inference ---###

In [ ]:
# import rank method via liana
#Rank_Aggregate method that combines the predictions of multiple ligand-receptor method
from liana.mt import rank_aggregate
import liana as li

In [ ]:
### LIANA+
li.method.show_methods()

In [ ]:
# Annotation
sample_key = 'sample'
condition_key = 'label'
groupby = 'cell_type_spec'

In [ ]:
# Ligand-Receptor Inference by Sample
li.mt.rank_aggregate.by_sample(
    adata,
    groupby=groupby,
    sample_key=condition_key, # sample key by which we which to loop
    expr_prop = 0.1,
    use_raw=False,
    n_perms=100,
    return_all_lrs=False,
    verbose=True, 
    )

In [ ]:
adata.uns["liana_res"].sort_values("magnitude_rank")

In [ ]:
adata.uns["liana_res"]['source'].unique()

In [ ]:
adata.uns["liana_res"]['ligand_complex'].unique()

In [ ]:
adata.uns["liana_res"]['receptor_complex'].unique()

In [ ]:
import plotnine as p9
my_plot = (li.pl.dotplot_by_sample(adata, sample_key=condition_key,
                         colour="magnitude_rank", size ="specificity_rank",
                         target_labels=[ 'SubLymphoid_11'],
                         source_labels=['SubMyeloid_0', 'SubMyeloid_1', 'SubMyeloid_5', 'SubMyeloid_6', 'SubMyeloid_14','SubMyeloid_3','SubMyeloid_8','SubMyeloid_4'],
                         ligand_complex=["LGALS9","LGALS3",	"HLA-DRB5", "HLA-DRA", "HLA-DRB1", "HLA-DQA1",  "CD274", "CALR", "ICAM1", "CD86", "CD80"],
                         receptor_complex=["HAVCR2", "LAG3", "CD80", "ITGAV", "IL2RA","CD28"],
                        
                         inverse_colour=True,
                         inverse_size=True,
                                              
                         #size_range=(0.5, 5),                         
                         ) +
    # rotate facet labels
   p9.theme_bw(base_size=14) + p9.scale_color_cmap('RdYlBu_r')+p9.theme(strip_text=p9.element_text(size=10, colour="black", angle=90), figure_size=(8, 20))
)

my_plot.save('dotplot_chat_liana.pdf')